# Manip 0 — Dimensionality of the Negation Subspace

**Question.** When a sentence is negated, its embedding moves by
$\Delta = e(\text{negated}) - e(\text{affirmative})$.
How many dimensions does that displacement actually live in?

The literature makes three *mutually incompatible* implicit assumptions, none of them ever tested:

| Source | Implicit assumption | Predicted geometry |
|---|---|---|
| Sammani et al. (negation steering) | one linear probe weight vector | **rank 1** — a single universal direction |
| Aggarwal et al. (*Seeing What's Not There*) | `e_neg` recomputed **per sentence** | effectively **unbounded** rank |
| Petcu et al. (negation taxonomy) | negation types differ strongly in difficulty | **one sub-axis per type** |

### Hypotheses tested here

| ID | Name | Signature in the data |
|---|---|---|
| **H1** | Pure translation | High translation share; significant rank $\approx 1$ |
| **H2** | Type-structured subspace | Significant rank $\approx$ #types; high intra-type coherence, low inter-type cosine |
| **H3** | Content-dependent subspace | Moderate rank, but **no** type structure |
| **H4** | No exploitable structure | Spectrum indistinguishable from the null model |

### What each outcome implies downstream

| Outcome | Correction operator | Cross-model transfer machinery |
|---|---|---|
| H1 | Single Householder reflection | Classic orthogonal Procrustes on one vector |
| H2 | Soft mixture of type subspaces | CCA + Procrustes (Agarwal's released pipeline) |
| H3 | Hybrid global subspace + local (attention) term | CCA + Procrustes on the global part only |
| H4 | Try kernel PCA / non-linear autoencoder first | Autoencoder mapping (Oozeer et al.) |

---
**Pipeline:** dataset registry → CLIP text embeddings (HuggingFace) → Δ vectors → spectrum →
information-theoretic dimension estimators → null models → cross-validated rank →
type structure → verdict.

**Which dataset** is a one-line choice in §0 (`CONFIG["dataset"]`); everything downstream reads the data
through the column map declared for it in §1, so no other cell changes:

* `"taxonomy"` — the purpose-built factorial set, `data/negation_dataset.csv`
* `"nevir"` — real-world IR queries, built by `00_load_nevir.ipynb`
* `"demo"` — a tiny hand-built set, pipeline smoke test only

## 0. Setup

Recommended environment (run once, outside the notebook):

```bash
python3 -m venv .venv && source .venv/bin/activate
pip install numpy pandas pyarrow torch transformers matplotlib scikit-learn ipykernel
```

In [ ]:
# !pip install -q numpy pandas pyarrow torch transformers matplotlib scikit-learn

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (7.0, 4.0),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 9,
})

CONFIG = {
    # ---- data -------------------------------------------------------------
    # The dataset to analyse. Entries are declared in the DATASETS registry (§1), which also
    # holds the file path, the reader and the column map — so switching dataset is this one line.
    #   "taxonomy" -> ../data/negation_dataset.csv   (18 categories x ~556 pairs, purpose-built)
    #   "nevir"    -> ../data/nevir_pairs.parquet    (produced by 00_load_nevir.ipynb)
    #   "demo"     -> tiny hand-built set, pipeline smoke test only
    "dataset": "taxonomy",
    "cache_dir": "../data/cache",
    "demo_mode_if_missing": True,      # fall back to the toy dataset if the file is absent

    "min_pairs_per_category": 5,       # categories rarer than this are dropped (unstable centroids)
    # Stratified cap on the pairs kept per category (None -> keep everything). Both the null models
    # (§6, one SVD per replicate) and the layer sweep (§11, one embedding pass per layer) scale with
    # N, so on the 10k-pair taxonomy dataset a full CPU run is tens of minutes; set this to e.g. 100
    # while iterating, then remove the cap for the final numbers.
    "max_pairs_per_category": None,

    # ---- model ------------------------------------------------------------
    "model_name": "openai/clip-vit-base-patch32",
    "layer": None,          # None -> final projected joint space; int -> text-encoder hidden layer
    "project_intermediate": False,   # apply final_layer_norm + text_projection to intermediate layers
    "batch_size": 128,
    "device": None,         # None -> auto (mps / cuda / cpu)

    # ---- delta construction ----------------------------------------------
    # "normalized": L2-normalise each embedding before subtracting (recommended for CLIP,
    #               whose cosine geometry is the operative one)
    # "raw":        subtract raw embeddings
    "delta_mode": "normalized",
    "unit_delta": False,    # additionally L2-normalise each delta (kills magnitude, keeps direction)

    # ---- analysis ---------------------------------------------------------
    "variance_thresholds": [0.80, 0.90, 0.95, 0.99],
    "n_null": 200,          # bootstrap replicates for the null spectra
    "null_quantile": 0.95,
    "cv_folds": 5,
    "max_rank_plot": 40,
    "seed": 0,

    # ---- verdict thresholds ----------------------------------------------
    "h1_translation_share": 0.50,   # above this, the mean displacement dominates
    "h1_max_rank": 2,
    "h2_max_rank": 12,
    "h2_min_intra_coherence": 0.45,
    "h2_max_inter_cosine": 0.60,
}

rng = np.random.default_rng(CONFIG["seed"])
Path(CONFIG["cache_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps({k: v for k, v in CONFIG.items() if k != "variance_thresholds"}, indent=2, default=str))

## 1. Dataset

One dataset is analysed at a time, chosen by `CONFIG["dataset"]`. The `DATASETS` registry below declares,
for each one, **where the file is, how to read it, and how its columns map onto the internal schema**.
Every later cell reads the data through `COLS`, never through a hard-coded column name, so adding a
dataset costs exactly one registry entry.

### Internal schema and where each dataset provides it

| Internal name | Required | Meaning | `taxonomy` | `nevir` | `demo` |
|---|---|---|---|---|---|
| `affirmative` | **yes** | Affirmative sentence | `affirmative` | `affirmative` | `affirmative` |
| `negative` | **yes** | Negated counterpart (minimal edit) | `negated` | `negative` | `negative` |
| `category` | **yes** | Negation mechanism | `category` | `category` | `category` |
| `pair_id` | no | Identifier of the minimal pair | `id` | `pair_id` | `pair_id` |
| `subclass` | no | Finer cue inside the category (`isn't` vs `is not`) | `marker` | `subclass` | `subclass` |
| `superclass` | no | Coarser grouping of categories | `superclass` | — | — |
| `object` | no | Controlled content word — enables the **category × object** decomposition (§8b) | — | `object` (source passage) | `object` |
| `scope` | no | `sentential` / `local` | — | `scope` | `scope` |
| `is_distractor` | no | `True` for morphological controls (`mis-`, `anti-`) that are **not** semantic negation | — | `is_distractor` | `is_distractor` |

A missing optional column is filled with a default and **the sections that need it skip themselves**, with
a printed message rather than an error. On `taxonomy` that means §8b (category × object decomposition) and
§9 (distractor check) do not run: the dataset has neither a content-control field nor `mis-`/`anti-` rows.

### The three datasets

* **`taxonomy`** — 10 000 pairs generated from the project taxonomy: 18 negation categories balanced at
  ~556 pairs each, nested in 8 superclasses, 143 distinct markers. Orientation is correct by construction
  (the generator writes each member into its own column), so unlike NevIR there is no sign ambiguity in Δ
  and the mean displacement — the whole H1 signal — is measured without attenuation.
* **`nevir`** — real-world IR queries harvested and auto-classified by `00_load_nevir.ipynb`. Heavily
  imbalanced across mechanisms, orientation inferred rather than given, and `other_substitution` is not
  negation at all (antonyms and rephrasings the classifier could not localise, arbitrary orientation) —
  pooling it in would inflate the rank with generic lexical variation, so the registry excludes it.
* **`demo`** — a tiny hand-built set. It exists only to check that the pipeline runs end to end; its
  numbers are meaningless because N is far below the embedding dimension.

Where they exist, the distractor categories are the cheapest validity check in the whole protocol: if
`mis-`/`anti-` land inside the negation subspace, the geometry is tracking a surface prefix cue, not
negation semantics.

In [ ]:
# --- registry -------------------------------------------------------------------
# `cols` maps INTERNAL name -> column name as it appears in that file. Everything downstream
# goes through COLS, so a new dataset is one entry here and nothing else.
STD_COLS = {"pair_id": "pair_id", "affirmative": "affirmative", "negative": "negative",
            "category": "category", "subclass": "subclass", "superclass": "superclass",
            "object": "object", "scope": "scope", "is_distractor": "is_distractor"}

DATASETS = {
    "taxonomy": {
        "path": "../data/negation_dataset.csv",
        "reader": pd.read_csv,
        "cols": {**STD_COLS, "pair_id": "id", "negative": "negated", "subclass": "marker"},
        "exclude_categories": [],
    },
    "nevir": {
        "path": "../data/nevir_pairs.parquet",
        "reader": pd.read_parquet,
        "cols": dict(STD_COLS),
        # `other_substitution` is NOT negation: those pairs differ by an antonym or a rephrasing the
        # classifier could not localise, and their orientation is arbitrary — pooling them in would
        # inflate the rank with generic lexical variation, the exact artefact we are avoiding.
        "exclude_categories": ["other_substitution"],
    },
    "demo": {"path": None, "reader": None, "cols": dict(STD_COLS), "exclude_categories": []},
}

# --- toy dataset ----------------------------------------------------------------
DEMO_OBJECTS = ["cat", "table", "window", "car", "tree", "book", "phone", "chair", "door", "lamp"]

DEMO_ADJ_PREFIX = [  # (positive, negated-by-prefix, subclass)
    ("possible", "impossible", "im_allomorph"), ("happy", "unhappy", "un"),
    ("correct", "incorrect", "in"), ("visible", "invisible", "in"),
    ("legal", "illegal", "il_allomorph"), ("regular", "irregular", "ir_allomorph"),
    ("clear", "unclear", "un"), ("accurate", "inaccurate", "in"),
    ("consistent", "inconsistent", "in"), ("able", "unable", "un"),
]
DEMO_NOUN_LESS = [  # (noun, -less adjective)
    ("meaning", "meaningless"), ("care", "careless"), ("hope", "hopeless"),
    ("use", "useless"), ("harm", "harmless"), ("fear", "fearless"),
    ("taste", "tasteless"), ("color", "colorless"), ("end", "endless"), ("power", "powerless"),
]
DEMO_IMPLICIT = [  # (affirmative verb phrase, implicitly negative verb phrase)
    ("managed to finish", "failed to finish"), ("accepted the offer", "rejected the offer"),
    ("agreed to help", "refused to help"), ("confirmed the report", "denied the report"),
    ("included the file", "excluded the file"), ("allowed the change", "forbade the change"),
    ("kept the receipt", "lost the receipt"), ("approached the gate", "avoided the gate"),
    ("mentioned the issue", "omitted the issue"), ("trusted the claim", "doubted the claim"),
]
DEMO_ANTONYM = [
    ("fast", "slow"), ("big", "small"), ("hot", "cold"), ("tall", "short"), ("heavy", "light"),
    ("bright", "dark"), ("cheap", "expensive"), ("easy", "hard"), ("wet", "dry"), ("full", "empty"),
]
DEMO_MIS = [  # DISTRACTOR: "wrongly", not "not"
    ("understood", "misunderstood"), ("led", "misled"), ("placed", "misplaced"),
    ("judged", "misjudged"), ("read", "misread"), ("calculated", "miscalculated"),
    ("interpreted", "misinterpreted"), ("spelled", "misspelled"),
    ("handled", "mishandled"), ("labeled", "mislabeled"),
]
DEMO_ANTI = [  # DISTRACTOR: oppositional, not absence
    ("social", "antisocial"), ("productive", "counterproductive"), ("viral", "antiviral"),
    ("clockwise", "counterclockwise"), ("bacterial", "antibacterial"), ("intuitive", "counterintuitive"),
    ("septic", "antiseptic"), ("climactic", "anticlimactic"), ("freeze", "antifreeze"),
    ("magnetic", "antimagnetic"),
]


def build_demo_dataframe(cols: dict) -> pd.DataFrame:
    """Tiny hand-built dataset so the whole pipeline can be smoke-tested without any file."""
    rows = []

    def add(cat, sub, obj, aff, neg, distractor=False, scope="sentential"):
        rows.append({
            cols["pair_id"]: f"{cat}_{len(rows):04d}", cols["category"]: cat, cols["subclass"]: sub,
            cols["object"]: obj, cols["scope"]: scope, cols["is_distractor"]: distractor,
            cols["affirmative"]: aff, cols["negative"]: neg,
        })

    for o in DEMO_OBJECTS:
        add("particle_not", "copula", o, f"The photo shows a {o}.", f"The photo does not show a {o}.")
        add("quantifier_no", "incorporated", o, f"There is a {o} in the room.", f"There is no {o} in the room.")
    for pos, neg, sub in DEMO_ADJ_PREFIX:
        add("prefix_un_in", sub, pos, f"The result is {pos}.", f"The result is {neg}.")
    for noun, adj in DEMO_NOUN_LESS:
        add("suffix_less", "less", noun, f"The plan has {noun}.", f"The plan is {adj}.")
    for pos, neg in DEMO_IMPLICIT:
        add("implicit_verb", "implicative", pos.split()[0], f"The engineer {pos}.", f"The engineer {neg}.")
    for pos, neg in DEMO_ANTONYM:
        add("antonym_polar", "polar", pos, f"The train is {pos}.", f"The train is {neg}.")
    for pos, neg in DEMO_MIS:
        add("mis_prefix", "mis", pos, f"The analyst {pos} the message.", f"The analyst {neg} the message.", True)
    for pos, neg in DEMO_ANTI:
        add("anti_prefix", "anti", pos, f"The compound is {pos}.", f"The compound is {neg}.", True)

    return pd.DataFrame(rows)


# --- loading --------------------------------------------------------------------
def load_dataset(name: str):
    """Return (frame, cols, resolved_name, is_demo) for a registry entry, falling back to the toy set."""
    if name not in DATASETS:
        raise KeyError(f"unknown dataset {name!r} — available: {list(DATASETS)}")
    spec = DATASETS[name]

    if spec["path"] is None:                                   # the demo entry itself
        return build_demo_dataframe(spec["cols"]), spec["cols"], name, True

    path = Path(spec["path"])
    if path.exists():
        frame = spec["reader"](path)
        print(f"dataset '{name}': loaded {len(frame)} pairs from {path}")
        return frame, spec["cols"], name, False

    if not CONFIG["demo_mode_if_missing"]:
        raise FileNotFoundError(f"{path} not found (dataset '{name}') and demo mode disabled.")
    print("=" * 78)
    print(f"DEMO MODE — {path} not found, falling back to the small hand-built dataset.")
    print("Numbers below are NOT meaningful (N is far too small relative to the embedding")
    print("dimension); this only validates that the pipeline runs end to end.")
    print("=" * 78)
    demo = DATASETS["demo"]
    return build_demo_dataframe(demo["cols"]), demo["cols"], "demo", True


df, COLS, DATASET, DEMO = load_dataset(CONFIG["dataset"])
SPEC = DATASETS[DATASET]

# --- schema validation ---------------------------------------------------------
required = [COLS["affirmative"], COLS["negative"], COLS["category"]]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required column(s): {missing}. Present: {list(df.columns)}")

for optional, default in [(COLS["is_distractor"], False), (COLS["subclass"], "na"), (COLS["scope"], "na")]:
    if optional not in df.columns:
        df[optional] = default

df = df.dropna(subset=required).reset_index(drop=True)

# --- filtering -----------------------------------------------------------------
if SPEC["exclude_categories"]:
    drop = df[COLS["category"]].isin(SPEC["exclude_categories"])
    if drop.any():
        print(f"dropped {int(drop.sum())} pairs from excluded categories: {SPEC['exclude_categories']}")
    df = df[~drop].reset_index(drop=True)

if CONFIG["min_pairs_per_category"] > 1:
    counts = df[COLS["category"]].value_counts()
    rare = counts[counts < CONFIG["min_pairs_per_category"]].index.tolist()
    if rare:
        print(f"dropped {len(rare)} category(ies) with < {CONFIG['min_pairs_per_category']} pairs: {rare}")
    df = df[~df[COLS["category"]].isin(rare)].reset_index(drop=True)

# Stratified subsample: shuffle, keep the first `cap` rows of each category. Preserves the category
# balance (the point of the taxonomy dataset) while cutting the cost of §6 and §11.
cap = CONFIG["max_pairs_per_category"]
if cap:
    before = len(df)
    df = (df.sample(frac=1.0, random_state=CONFIG["seed"])
            .groupby(COLS["category"], sort=False).head(cap)
            .sort_index().reset_index(drop=True))
    print(f"subsampled {before} -> {len(df)} pairs (cap = {cap} per category)")

HAS_OBJECT = COLS["object"] in df.columns and df[COLS["object"]].notna().all()
HAS_SUPERCLASS = COLS["superclass"] in df.columns and df[COLS["superclass"]].notna().all()
N_DISTRACTOR = int(df[COLS["is_distractor"]].astype(bool).sum())

print(f"\ndataset '{DATASET}' | {len(df)} pairs | {df[COLS['category']].nunique()} categories | "
      f"object column: {HAS_OBJECT} | distractor rows: {N_DISTRACTOR}")
if not HAS_OBJECT:
    print("  -> §8b (category × object variance decomposition) will be skipped")
if N_DISTRACTOR == 0:
    print("  -> §9 (mis-/anti- distractor validity check) will be skipped")

group_by = ([COLS["superclass"]] if HAS_SUPERCLASS else []) + [COLS["category"], COLS["is_distractor"]]
display(df.groupby(group_by).size().rename("n").reset_index())
display(df.head(8)[[COLS["category"], COLS["subclass"], COLS["affirmative"], COLS["negative"]]])

## 2. CLIP text embeddings (HuggingFace)

Two extraction modes:

* `layer=None` → the **final projected joint space** (`get_text_features`), i.e. the space in which
  image–text cosine similarity is computed. This is the space Aggarwal et al. operate in.
* `layer=k` → the hidden state of the **EOS token** at text-encoder layer `k`. Sammani et al. report
  that negation is best linearly encoded in *intermediate* layers, so the layer sweep in §10 matters.

In [ ]:
import torch
from transformers import AutoTokenizer, CLIPModel


def pick_device(requested=None) -> str:
    if requested:
        return requested
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


DEVICE = pick_device(CONFIG["device"])
print(f"device = {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
model = CLIPModel.from_pretrained(CONFIG["model_name"]).to(DEVICE).eval()
N_TEXT_LAYERS = model.config.text_config.num_hidden_layers
print(f"{CONFIG['model_name']} | text layers = {N_TEXT_LAYERS} | "
      f"width = {model.config.text_config.hidden_size} | projection = {model.config.projection_dim}")


def _as_tensor(out):
    """transformers <5 returns a tensor from get_text_features; >=5 returns an output object
    whose `pooler_output` holds the projected joint-space embedding (verified identical to
    text_projection(final_layer_norm(EOS hidden state)))."""
    if isinstance(out, torch.Tensor):
        return out
    for attr in ("text_embeds", "pooler_output"):
        v = getattr(out, attr, None)
        if v is not None:
            return v
    return out[0]


def _eos_positions(input_ids: torch.Tensor) -> torch.Tensor:
    """Index of the EOS token per sequence (CLIP pools the sentence representation there)."""
    eos_id = tokenizer.eos_token_id
    if eos_id is None:
        return input_ids.argmax(dim=-1)
    hit = input_ids == eos_id
    return torch.where(hit.any(dim=-1), hit.int().argmax(dim=-1), input_ids.argmax(dim=-1))


@torch.no_grad()
def embed_texts(texts, layer=None, batch_size=None, project_intermediate=None) -> np.ndarray:
    """Return (N, d) float32 embeddings. layer=None -> final projected joint space."""
    batch_size = batch_size or CONFIG["batch_size"]
    project_intermediate = CONFIG["project_intermediate"] if project_intermediate is None else project_intermediate
    texts = list(texts)
    out = []

    for start in range(0, len(texts), batch_size):
        chunk = texts[start:start + batch_size]
        enc = tokenizer(chunk, padding=True, truncation=True, max_length=77, return_tensors="pt").to(DEVICE)

        if layer is None:
            feats = _as_tensor(model.get_text_features(input_ids=enc["input_ids"],
                                                       attention_mask=enc["attention_mask"]))
        else:
            res = model.text_model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                                   output_hidden_states=True)
            hs = res.hidden_states[layer]                       # (B, T, width)
            pos = _eos_positions(enc["input_ids"])
            feats = hs[torch.arange(hs.shape[0], device=hs.device), pos]
            if project_intermediate:
                feats = model.text_projection(model.text_model.final_layer_norm(feats))

        out.append(feats.float().cpu().numpy())

    return np.concatenate(out, axis=0)


def cache_key(layer) -> str:
    payload = "|".join([DATASET, CONFIG["model_name"], str(layer), str(CONFIG["project_intermediate"]),
                        str(len(df)), str(df[COLS["affirmative"]].iloc[0]),
                        str(df[COLS["negative"]].iloc[-1])])
    return hashlib.md5(payload.encode()).hexdigest()[:12]


def get_embeddings(layer=None, use_cache=True):
    """Embed affirmative and negative sentences, with an on-disk cache."""
    path = Path(CONFIG["cache_dir"]) / f"emb_{cache_key(layer)}.npz"
    if use_cache and path.exists():
        z = np.load(path)
        return z["aff"], z["neg"]
    aff = embed_texts(df[COLS["affirmative"]].tolist(), layer=layer)
    neg = embed_texts(df[COLS["negative"]].tolist(), layer=layer)
    if use_cache:
        np.savez_compressed(path, aff=aff, neg=neg)
    return aff, neg


E_AFF, E_NEG = get_embeddings(CONFIG["layer"])
N, D = E_AFF.shape
print(f"embeddings: aff {E_AFF.shape} | neg {E_NEG.shape}")
if N < 5 * D:
    print(f"\nNOTE  N={N} vs d={D}: the sample is small relative to the dimension, so raw PCA")
    print("      eigenvalues are inflated by sampling noise. The null models in §5 and the")
    print("      cross-validated rank in §6 are the estimates to trust, not the scree plot.")

## 3. Δ vectors — translation share vs. residual structure

**This is the subtlety that decides whether the analysis is even valid.**

* The **mean** $\bar\Delta$ *is* the candidate translation vector of H1.
* The **covariance** of $\Delta$ (i.e. after centering) describes the variability *around* that translation.

So running PCA only on centered Δ would remove exactly the H1 signal before measuring it.
We therefore report both:

1. **Translation share** $\;\rho = \dfrac{\lVert \bar\Delta \rVert^2}{\mathbb{E}\lVert \Delta \rVert^2} \in [0,1]$ —
   the fraction of displacement energy carried by the common direction. $\rho \to 1$ means pure translation.
2. The **uncentered second-moment** spectrum — what a correction operator would actually have to model.
3. The **centered covariance** spectrum — the structure of the deviations (this is where negation *types* would live).

In [ ]:
def l2n(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), eps)


def build_deltas(e_aff, e_neg, mode=None, unit=None) -> np.ndarray:
    mode = mode or CONFIG["delta_mode"]
    unit = CONFIG["unit_delta"] if unit is None else unit
    a, b = (l2n(e_aff), l2n(e_neg)) if mode == "normalized" else (e_aff, e_neg)
    d = b - a
    return l2n(d) if unit else d


DELTA = build_deltas(E_AFF, E_NEG)

mean_delta = DELTA.mean(axis=0)
translation_share = float(np.dot(mean_delta, mean_delta) / np.mean(np.sum(DELTA ** 2, axis=1)))

Dn = l2n(DELTA)
cos_to_mean = Dn @ l2n(mean_delta)
sample = rng.choice(N, size=min(N, 400), replace=False)
pair_cos = (Dn[sample] @ Dn[sample].T)[np.triu_indices(len(sample), k=1)]

print(f"translation share  rho = {translation_share:.3f}")
print(f"mean cos(delta_i, mean delta) = {cos_to_mean.mean():.3f}")
print(f"mean pairwise cos(delta_i, delta_j) = {pair_cos.mean():.3f}")
print(f"mean ||delta|| = {np.linalg.norm(DELTA, axis=1).mean():.4f}")

fig, ax = plt.subplots(1, 3, figsize=(11, 3.1))
ax[0].hist(np.linalg.norm(DELTA, axis=1), bins=40, color="#4C72B0")
ax[0].set(title="||Δ|| distribution", xlabel="norm", ylabel="count")
ax[1].hist(cos_to_mean, bins=40, color="#DD8452")
ax[1].axvline(0, color="k", lw=0.8)
ax[1].set(title="cos(Δᵢ, mean Δ)", xlabel="cosine")
ax[2].hist(pair_cos, bins=40, color="#55A868")
ax[2].axvline(0, color="k", lw=0.8)
ax[2].set(title="pairwise cos(Δᵢ, Δⱼ)", xlabel="cosine")
fig.suptitle(f"Displacement coherence — translation share ρ = {translation_share:.3f}", y=1.04)
fig.tight_layout()
plt.show()

## 4. Spectrum: second moment (uncentered) and covariance (centered)

In [ ]:
@dataclass
class Spectrum:
    eigenvalues: np.ndarray                 # descending
    name: str = ""
    components: np.ndarray | None = None    # (k, d) right singular vectors

    @property
    def p(self) -> np.ndarray:
        """Normalised spectrum — a probability distribution over directions."""
        ev = np.clip(self.eigenvalues, 0, None)
        s = ev.sum()
        return ev / s if s > 0 else ev

    @property
    def cumulative(self) -> np.ndarray:
        return np.cumsum(self.p)


def spectrum_of(X: np.ndarray, center: bool, name: str = "", keep: int | None = None) -> Spectrum:
    """Eigen-spectrum of the (co)variance of the rows of X, computed through the SVD."""
    M = X - X.mean(axis=0, keepdims=True) if center else X
    denom = max(len(M) - 1, 1) if center else len(M)
    U, s, Vt = np.linalg.svd(M, full_matrices=False)
    ev = (s ** 2) / denom
    if keep is not None:
        ev, Vt = ev[:keep], Vt[:keep]
    return Spectrum(eigenvalues=ev, name=name, components=Vt)


SPEC_UNCENTERED = spectrum_of(DELTA, center=False, name="second moment (uncentered)")
SPEC_CENTERED = spectrum_of(DELTA, center=True, name="covariance (centered)")

kmax = min(CONFIG["max_rank_plot"], len(SPEC_UNCENTERED.eigenvalues))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for a, spec in zip(ax, [SPEC_UNCENTERED, SPEC_CENTERED]):
    a.plot(np.arange(1, kmax + 1), spec.p[:kmax], "o-", ms=3.5, lw=1.2)
    a.set_yscale("log")
    a.set(title=f"Scree — {spec.name}", xlabel="component", ylabel="explained variance ratio")
fig.tight_layout()
plt.show()

print(f"top-1 share  uncentered = {SPEC_UNCENTERED.p[0]:.3f} | centered = {SPEC_CENTERED.p[0]:.3f}")
print(f"top-5 share  uncentered = {SPEC_UNCENTERED.cumulative[4]:.3f} | centered = {SPEC_CENTERED.cumulative[4]:.3f}")

## 5. Information-theoretic dimension estimators

Normalise the spectrum into a distribution $p_i = \lambda_i / \sum_j \lambda_j$. Then:

* **Shannon effective rank** (Roy & Vetterli, 2007): $\;\mathrm{erank} = \exp\big(-\sum_i p_i \log p_i\big)$.
  The canonical information-based dimension: the exponential of the entropy of the spectrum.
* **Participation ratio**: $\;\mathrm{PR} = \big(\sum_i \lambda_i\big)^2 / \sum_i \lambda_i^2 = 1/\sum_i p_i^2$.
  This is $\exp(H_2)$, the Rényi-2 effective rank — less sensitive to the long tail of small eigenvalues.
* **Rényi family**: $\;\mathrm{erank}_\alpha = \big(\sum_i p_i^\alpha\big)^{1/(1-\alpha)}$, with $\alpha \to 1$
  giving Shannon and $\alpha = 2$ giving PR. Sweeping $\alpha$ shows how much the estimate depends on the
  weight given to small eigenvalues — a **flat** curve means the dimension estimate is robust, a steeply
  decreasing one means "the answer depends on where you cut the tail", which is itself diagnostic.
* **Variance thresholds**: $r(\tau) = \min\{k : \sum_{i \le k} p_i \ge \tau\}$.
* **Elbow**: maximum-curvature point of the cumulative curve.

In [ ]:
def shannon_effective_rank(p: np.ndarray) -> float:
    q = p[p > 1e-15]
    return float(np.exp(-(q * np.log(q)).sum()))


def participation_ratio(ev: np.ndarray) -> float:
    ev = np.clip(ev, 0, None)
    denom = (ev ** 2).sum()
    return float((ev.sum() ** 2) / denom) if denom > 0 else 0.0


def renyi_effective_rank(p: np.ndarray, alpha: float) -> float:
    q = p[p > 1e-15]
    if abs(alpha - 1.0) < 1e-9:
        return shannon_effective_rank(q)
    return float((q ** alpha).sum() ** (1.0 / (1.0 - alpha)))


def rank_at_threshold(spec: Spectrum, tau: float) -> int:
    return int(np.searchsorted(spec.cumulative, tau) + 1)


def elbow_rank(spec: Spectrum, kmax: int | None = None) -> int:
    """Maximum-curvature index of the cumulative explained-variance curve."""
    c = spec.cumulative[:kmax] if kmax else spec.cumulative
    if len(c) < 4:
        return len(c)
    return int(np.argmax(-np.diff(c, n=2)) + 2)


def estimator_table(spec: Spectrum) -> pd.DataFrame:
    rows = [
        ("Shannon effective rank  exp(H)", shannon_effective_rank(spec.p)),
        ("Participation ratio     exp(H2)", participation_ratio(spec.eigenvalues)),
        ("Renyi rank  alpha=0.5", renyi_effective_rank(spec.p, 0.5)),
        ("Renyi rank  alpha=3", renyi_effective_rank(spec.p, 3.0)),
        ("Elbow (max curvature)", elbow_rank(spec)),
    ]
    rows += [(f"Variance threshold  tau={t:.2f}", rank_at_threshold(spec, t))
             for t in CONFIG["variance_thresholds"]]
    return pd.DataFrame(rows, columns=["estimator", spec.name]).set_index("estimator")


EST = estimator_table(SPEC_UNCENTERED).join(estimator_table(SPEC_CENTERED))
display(EST.round(2))

alphas = np.linspace(0.5, 4.0, 36)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for spec, style in [(SPEC_UNCENTERED, "-"), (SPEC_CENTERED, "--")]:
    ax[0].plot(np.arange(1, kmax + 1), spec.cumulative[:kmax], style, lw=1.4, label=spec.name)
    ax[1].plot(alphas, [renyi_effective_rank(spec.p, a) for a in alphas], style, lw=1.4, label=spec.name)
for t in CONFIG["variance_thresholds"]:
    ax[0].axhline(t, color="grey", lw=0.6, ls=":")
    ax[0].text(kmax * 0.99, t + 0.008, f"τ={t:.2f}", ha="right", fontsize=7, color="grey")
ax[0].set(title="Cumulative explained variance", xlabel="rank r", ylabel="cumulative share", ylim=(0, 1.02))
ax[0].legend(fontsize=7)
ax[1].axvline(1.0, color="grey", lw=0.6, ls=":")
ax[1].axvline(2.0, color="grey", lw=0.6, ls=":")
ax[1].text(1.02, ax[1].get_ylim()[1] * 0.92, "Shannon", fontsize=7, color="grey")
ax[1].text(2.02, ax[1].get_ylim()[1] * 0.82, "PR", fontsize=7, color="grey")
ax[1].set(title="Rényi effective rank vs α", xlabel="α", ylabel="effective rank")
ax[1].legend(fontsize=7)
fig.tight_layout()
plt.show()

## 6. Null models — which components are real?

Even perfectly structureless vectors produce a decaying spectrum when $N \lesssim d$
(Marchenko–Pastur regime). A raw scree plot therefore proves nothing on its own.
Three nulls, from most to least conservative:

| Null | Construction | What it controls for |
|---|---|---|
| `shuffled` | $\Delta = e(\text{neg}_j) - e(\text{aff}_i),\; i \ne j$ | pairing — keeps both marginal distributions |
| `affirmative` | $\Delta = e(\text{aff}_i) - e(\text{aff}_j)$ | **generic semantic difference** between two unrelated sentences |
| `gaussian` | isotropic Gaussian, matched $N, d$, matched total energy | pure sampling noise (the MP baseline) |

`affirmative` is the scientifically interesting one: it answers *"is negation lower-dimensional than an
arbitrary semantic difference?"* rather than merely *"is it lower-dimensional than noise?"*.
Spectra are compared **after normalisation** (shape, not magnitude), and the **significant rank** is the
number of leading components exceeding the null's per-index quantile.

In [ ]:
def null_deltas(kind: str, n: int) -> np.ndarray:
    a, b = (l2n(E_AFF), l2n(E_NEG)) if CONFIG["delta_mode"] == "normalized" else (E_AFF, E_NEG)
    if kind == "shuffled":
        i = rng.integers(0, N, size=n)
        j = rng.integers(0, N, size=n)
        j = np.where(j == i, (j + 1) % N, j)
        d = b[j] - a[i]
    elif kind == "affirmative":
        i = rng.integers(0, N, size=n)
        j = rng.integers(0, N, size=n)
        j = np.where(j == i, (j + 1) % N, j)
        d = a[i] - a[j]
    elif kind == "matched":
        # Content-MATCHED, negation-free control: two affirmative sentences drawn from the same
        # content group (`object`). This is the fair null — "affirmative" pairs unrelated sentences,
        # whose difference is dominated by topic, which is not what we want to compare negation against.
        i, j = _matched_index_pairs(n)
        d = a[i] - a[j]
    elif kind == "gaussian":
        scale = np.sqrt(np.mean(np.sum(DELTA ** 2, axis=1)) / D)
        d = rng.normal(0.0, scale, size=(n, D))
    else:
        raise ValueError(kind)
    return l2n(d) if CONFIG["unit_delta"] else d


def _matched_index_pairs(n: int):
    """Sample (i, j) pairs of distinct rows sharing the same `object` (content) group."""
    groups = df[COLS["object"]].astype(str).values
    buckets = [np.flatnonzero(groups == g) for g in np.unique(groups)]
    buckets = [b for b in buckets if len(b) >= 2]
    if not buckets:
        raise RuntimeError("no content group has >= 2 pairs — 'matched' null unavailable")
    pick = rng.integers(0, len(buckets), size=n)
    i = np.empty(n, dtype=int)
    j = np.empty(n, dtype=int)
    for k, b in enumerate(pick):
        bucket = buckets[b]
        i[k], j[k] = rng.choice(bucket, size=2, replace=False)
    return i, j


def null_spectra(kind: str, n_rep: int | None = None, center: bool = False) -> np.ndarray:
    n_rep = n_rep or CONFIG["n_null"]
    keep = min(CONFIG["max_rank_plot"], N, D)
    out = np.zeros((n_rep, keep))
    for r in range(n_rep):
        spec = spectrum_of(null_deltas(kind, N), center=center, keep=keep)
        out[r] = spec.p[:keep]
    return out


NULL_KINDS = ["shuffled", "affirmative", "gaussian"]
if HAS_OBJECT:
    try:
        _matched_index_pairs(4)
        NULL_KINDS.insert(2, "matched")
    except RuntimeError as e:
        print(f"matched null unavailable: {e}")

NULLS = {k: null_spectra(k) for k in NULL_KINDS}
real_p = SPEC_UNCENTERED.p[:kmax]

sig_rank = {}
for kind, arr in NULLS.items():
    thr = np.quantile(arr[:, :kmax], CONFIG["null_quantile"], axis=0)
    above = real_p > thr
    sig_rank[kind] = int(np.argmin(above)) if not above.all() else int(kmax)  # first index failing the test

print(f"Significant rank (# leading components above the {CONFIG['null_quantile']:.0%} null quantile):")
for k, v in sig_rank.items():
    print(f"  vs {k:12s} -> {v}")

# Prefer the content-matched null when available: it isolates negation from topic variation.
REFERENCE_NULL = "matched" if "matched" in sig_rank else "affirmative"
SIGNIFICANT_RANK = sig_rank[REFERENCE_NULL]
print(f"\nreference null = {REFERENCE_NULL}  ->  significant rank = {SIGNIFICANT_RANK}")

fig, ax = plt.subplots(figsize=(7.6, 4.0))
x = np.arange(1, kmax + 1)
colors = {"shuffled": "#DD8452", "affirmative": "#55A868", "matched": "#C44E52", "gaussian": "#8C8C8C"}
for kind, arr in NULLS.items():
    lo, hi = np.quantile(arr[:, :kmax], [0.05, CONFIG["null_quantile"]], axis=0)
    ax.fill_between(x, lo, hi, alpha=0.22, color=colors[kind], label=f"null: {kind} (5–{CONFIG['null_quantile']:.0%})")
ax.plot(x, real_p, "o-", color="#4C72B0", ms=3.5, lw=1.4, label="observed Δ (negation)")
ax.axvline(SIGNIFICANT_RANK + 0.5, color="k", ls="--", lw=0.9)
ax.text(SIGNIFICANT_RANK + 0.7, real_p.max(), f"significant rank = {SIGNIFICANT_RANK}", fontsize=8)
ax.set_yscale("log")
ax.set(title="Observed spectrum vs null models (normalised)", xlabel="component", ylabel="explained variance ratio")
ax.legend(fontsize=7)
fig.tight_layout()
plt.show()

## 7. Cross-validated (predictive) rank

Sections 4–6 are **descriptive**: a PCA always describes its own training sample well.
The stricter question is *how many components generalise to pairs never seen*.

Fit $V_r$ (top-$r$ right singular vectors) on a training split, then on held-out Δ measure
$R^2(r) = 1 - \lVert \Delta - V_rV_r^\top\Delta\rVert^2 / \lVert\Delta\rVert^2$.
A random $r$-dimensional subspace already achieves $R^2 \approx r/d$, so the quantity that matters is the
**excess over random**. The plateau of the test curve is the predictive rank.

In [ ]:
def cv_reconstruction(X: np.ndarray, ranks, n_folds=None, seed=0):
    n_folds = n_folds or CONFIG["cv_folds"]
    ranks = np.asarray(ranks)
    idx = np.random.default_rng(seed).permutation(len(X))
    folds = np.array_split(idx, n_folds)
    r2_test = np.zeros((n_folds, len(ranks)))
    r2_train = np.zeros((n_folds, len(ranks)))
    r2_rand = np.zeros((n_folds, len(ranks)))

    for f, fold in enumerate(folds):
        test = X[fold]
        train = X[np.setdiff1d(idx, fold)]
        _, _, Vt = np.linalg.svd(train, full_matrices=False)
        Q, _ = np.linalg.qr(np.random.default_rng(seed + f).normal(size=(X.shape[1], ranks.max())))
        for c, r in enumerate(ranks):
            Vr = Vt[:r].T
            r2_test[f, c] = 1 - ((test - test @ Vr @ Vr.T) ** 2).sum() / (test ** 2).sum()
            r2_train[f, c] = 1 - ((train - train @ Vr @ Vr.T) ** 2).sum() / (train ** 2).sum()
            Qr = Q[:, :r]
            r2_rand[f, c] = 1 - ((test - test @ Qr @ Qr.T) ** 2).sum() / (test ** 2).sum()

    return ranks, r2_train.mean(0), r2_test.mean(0), r2_test.std(0), r2_rand.mean(0)


RANKS = np.unique(np.clip(np.round(np.geomspace(1, min(CONFIG["max_rank_plot"], N - 2, D), 18)).astype(int), 1, None))
ranks, r2_tr, r2_te, r2_sd, r2_rd = cv_reconstruction(DELTA, RANKS, seed=CONFIG["seed"])
excess = r2_te - r2_rd
predictive_rank = int(ranks[np.argmax(excess)]) if len(ranks) else 1

cv_table = pd.DataFrame({"rank": ranks, "R2_train": r2_tr, "R2_test": r2_te,
                         "R2_random": r2_rd, "excess_over_random": excess})
display(cv_table.round(3))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(ranks, r2_tr, "o--", ms=3.5, lw=1.1, color="#4C72B0", label="train")
ax[0].errorbar(ranks, r2_te, yerr=r2_sd, fmt="o-", ms=3.5, lw=1.4, color="#C44E52", label="test")
ax[0].plot(ranks, r2_rd, ":", lw=1.2, color="grey", label="random subspace (≈ r/d)")
ax[0].set(title="Held-out reconstruction of Δ", xlabel="rank r", ylabel="R²", xscale="log")
ax[0].legend(fontsize=7)
ax[1].plot(ranks, excess, "o-", ms=3.5, lw=1.4, color="#55A868")
ax[1].axhline(0, color="k", lw=0.8)
ax[1].set(title="Excess R² over a random subspace", xlabel="rank r", ylabel="ΔR²", xscale="log")
fig.tight_layout()
plt.show()

print(f"rank maximising excess R² = {predictive_rank} (excess = {excess.max():.3f})")

## 8. Type structure — H2 vs H3

Three diagnostics, computed on **true negation categories only** (distractors handled in §9):

* **Intra-category coherence** — mean cosine of each Δ to its own category centroid.
  High ⇒ the category has a stable direction.
* **Inter-category cosine** — cosine between category centroids.
  High everywhere ⇒ all types collapse onto one axis (H1 in disguise);
  low ⇒ genuinely distinct sub-axes (H2).
* **Principal angles** between per-category subspaces — the subspace-level generalisation of the cosine.

In [ ]:
IS_DIST = df[COLS["is_distractor"]].astype(bool).values
CATS = df[COLS["category"]].values
true_cats = sorted(set(CATS[~IS_DIST]))

centroids, coherence, per_cat_rank, counts = {}, {}, {}, {}
for c in true_cats:
    m = (CATS == c) & (~IS_DIST)
    counts[c] = int(m.sum())
    Dc = DELTA[m]
    cen = Dc.mean(axis=0)
    centroids[c] = cen
    coherence[c] = float(np.mean(l2n(Dc) @ l2n(cen))) if np.linalg.norm(cen) > 0 else 0.0
    per_cat_rank[c] = participation_ratio(spectrum_of(Dc, center=False).eigenvalues) if counts[c] > 2 else np.nan

C = np.stack([l2n(centroids[c]) for c in true_cats])
COS = C @ C.T
off = COS[np.triu_indices(len(true_cats), k=1)]

summary = pd.DataFrame({
    "n": pd.Series(counts),
    "intra_coherence": pd.Series(coherence),
    "participation_ratio": pd.Series(per_cat_rank),
}).sort_values("intra_coherence", ascending=False)
display(summary.round(3))

MEAN_INTRA = float(np.mean(list(coherence.values())))
MEAN_INTER = float(off.mean()) if len(off) else np.nan
print(f"mean intra-category coherence = {MEAN_INTRA:.3f}")
print(f"mean inter-category centroid cosine = {MEAN_INTER:.3f}  (min {off.min():.3f} / max {off.max():.3f})")
print("\nReading: high intra + LOW inter  -> H2 (distinct sub-axes per type)")
print("         high intra + HIGH inter -> H1 in disguise (all types share one axis)")
print("         low intra               -> H3 (content dominates over type)")

# The taxonomy dataset has 18 categories, the demo 7: size the figure with the matrix and drop the
# per-cell annotations once they stop being readable.
n_cat = len(true_cats)
side = max(5.6, 0.40 * n_cat + 3.0)
tick_fs = 7 if n_cat <= 12 else 6
fig, ax = plt.subplots(figsize=(side, side * 0.86))
im = ax.imshow(COS, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(n_cat), true_cats, rotation=45, ha="right", fontsize=tick_fs)
ax.set_yticks(range(n_cat), true_cats, fontsize=tick_fs)
if n_cat <= 12:
    for i in range(n_cat):
        for j in range(n_cat):
            ax.text(j, i, f"{COS[i, j]:.2f}", ha="center", va="center", fontsize=6.5,
                    color="white" if abs(COS[i, j]) > 0.55 else "black")
ax.set_title("Cosine between category centroids")
ax.grid(False)
fig.colorbar(im, shrink=0.8)
fig.tight_layout()
plt.show()

In [ ]:
def principal_angles(A: np.ndarray, B: np.ndarray, r: int) -> np.ndarray:
    """Principal angles (degrees) between the top-r subspaces spanned by rows of A and B."""
    def basis(X):
        _, _, Vt = np.linalg.svd(X - 0.0, full_matrices=False)
        Q, _ = np.linalg.qr(Vt[:r].T)
        return Q
    s = np.linalg.svd(basis(A).T @ basis(B), compute_uv=False)
    return np.degrees(np.arccos(np.clip(s, -1.0, 1.0)))


R_SUB = 3
eligible = [c for c in true_cats if counts[c] > R_SUB + 1]
if len(eligible) >= 2:
    PA = np.full((len(eligible), len(eligible)), np.nan)
    for i, a in enumerate(eligible):
        for j, b in enumerate(eligible):
            ang = principal_angles(DELTA[(CATS == a) & ~IS_DIST], DELTA[(CATS == b) & ~IS_DIST], R_SUB)
            PA[i, j] = ang.mean()
    n_el = len(eligible)
    side = max(5.6, 0.40 * n_el + 3.0)
    tick_fs = 7 if n_el <= 12 else 6
    fig, ax = plt.subplots(figsize=(side, side * 0.86))
    im = ax.imshow(PA, cmap="viridis_r", vmin=0, vmax=90)
    ax.set_xticks(range(n_el), eligible, rotation=45, ha="right", fontsize=tick_fs)
    ax.set_yticks(range(n_el), eligible, fontsize=tick_fs)
    if n_el <= 12:
        for i in range(n_el):
            for j in range(n_el):
                ax.text(j, i, f"{PA[i, j]:.0f}", ha="center", va="center", fontsize=6.5, color="white")
    ax.set_title(f"Mean principal angle between top-{R_SUB} subspaces (degrees)")
    ax.grid(False)
    fig.colorbar(im, shrink=0.8)
    fig.tight_layout()
    plt.show()
    print("0° = identical subspaces, 90° = orthogonal. Small off-diagonal angles ⇒ shared structure (H1-like).")
else:
    print(f"Not enough pairs per category for subspace angles (need > {R_SUB + 1} each).")

### 8b. Variance decomposition: category vs. object

A rank $> 1$ can mean two very different things:
genuine **negation-type structure** (H2), or **residual content leaking into Δ** — the identity of the
negated object failing to cancel in the subtraction.

With a factorial `category × object` design we can separate them by decomposing the total displacement
energy into a category main effect, an object main effect, and a residual.
*(Exact for balanced designs; approximate otherwise — the code reports the balance.)*

**If the object effect dominates, re-run the whole analysis on object-centred Δ**
($\Delta'_i = \Delta_i - \overline{\Delta}_{\text{object}(i)}$) before concluding anything about types.

In [ ]:
if HAS_OBJECT:
    OBJ = df[COLS["object"]].astype(str).values
    mask = ~IS_DIST
    X, cat_v, obj_v = DELTA[mask], CATS[mask], OBJ[mask]
    grand = X.mean(axis=0)

    def between_ss(labels):
        ss = 0.0
        for lv in np.unique(labels):
            m = labels == lv
            ss += m.sum() * float(np.sum((X[m].mean(axis=0) - grand) ** 2))
        return ss

    ss_total = float(np.sum((X - grand) ** 2))
    ss_cat, ss_obj = between_ss(cat_v), between_ss(obj_v)
    ss_res = max(ss_total - ss_cat - ss_obj, 0.0)

    tab = pd.crosstab(cat_v, obj_v)
    balance = float(tab.values.min() / max(tab.values.max(), 1))
    dec = pd.DataFrame({
        "effect": ["category", "object", "residual"],
        "share_of_variance": [ss_cat / ss_total, ss_obj / ss_total, ss_res / ss_total],
    })
    display(dec.round(3))
    print(f"design balance (min/max cell count) = {balance:.2f}  "
          f"({'balanced' if balance > 0.8 else 'UNBALANCED — decomposition is approximate'})")

    fig, ax = plt.subplots(figsize=(5.4, 3.0))
    ax.barh(dec["effect"], dec["share_of_variance"], color=["#4C72B0", "#DD8452", "#BBBBBB"])
    ax.set(title="Displacement variance decomposition", xlabel="share of total variance", xlim=(0, 1))
    fig.tight_layout()
    plt.show()

    if ss_obj > ss_cat:
        print("\nWARNING  Object identity explains more than negation category.")
        print("         Object-centred Δ is computed below — prefer it for the type analysis.")
    DELTA_OBJ_CENTERED = DELTA.copy()
    for lv in np.unique(OBJ):
        m = OBJ == lv
        DELTA_OBJ_CENTERED[m] -= DELTA[m].mean(axis=0)
    spec_oc = spectrum_of(DELTA_OBJ_CENTERED[~IS_DIST], center=False, name="object-centred Δ")
    print(f"\nEffective rank after object-centring: "
          f"Shannon {shannon_effective_rank(spec_oc.p):.2f} | PR {participation_ratio(spec_oc.eigenvalues):.2f}")
else:
    print("No `object` column — skipping the category × object decomposition.")
    print("This test is what separates real type structure (H2) from content leakage (H3);")
    print("add a controlled `object` field to the generated dataset to enable it.")

## 9. Distractor validity check (`mis-`, `anti-`)

`misunderstand` ≠ `not understand` and `antisocial` ≠ `not social`: these prefixes *look* negative but are
not semantic negation. Fit the subspace on true negation categories only, then measure how much of each
category's displacement energy it captures: $\;\lVert P_r\Delta\rVert^2/\lVert\Delta\rVert^2$.

**Distractors scoring like true categories ⇒ the geometry is keying on a surface morphological cue**,
the same failure mode NegBench documents for ConCLIP (all "negative-looking" captions collapsing together).

In [ ]:
if IS_DIST.any():
    R_FIT = max(1, min(SIGNIFICANT_RANK, 8))
    _, _, Vt_true = np.linalg.svd(DELTA[~IS_DIST], full_matrices=False)
    P = Vt_true[:R_FIT].T @ Vt_true[:R_FIT]
    captured = np.sum((DELTA @ P) ** 2, axis=1) / np.maximum(np.sum(DELTA ** 2, axis=1), 1e-12)

    res = (pd.DataFrame({"category": CATS, "is_distractor": IS_DIST, "captured": captured})
           .groupby(["is_distractor", "category"])["captured"].agg(["mean", "std", "size"])
           .sort_values("mean", ascending=False))
    display(res.round(3))

    baseline = R_FIT / D
    print(f"random-subspace baseline = r/d = {baseline:.3f}   (fitted rank r = {R_FIT})")
    print(f"true categories mean  = {captured[~IS_DIST].mean():.3f}")
    print(f"distractors  mean     = {captured[IS_DIST].mean():.3f}")
    verdict_dist = ("PASS — distractors are clearly outside the negation subspace"
                    if captured[IS_DIST].mean() < 0.6 * captured[~IS_DIST].mean()
                    else "FAIL — distractors sit inside the subspace: likely a surface-form shortcut")
    print(f"\n{verdict_dist}")

    fig, ax = plt.subplots(figsize=(7.2, 3.4))
    order = res.reset_index().sort_values("mean", ascending=False)
    ax.bar(range(len(order)), order["mean"],
           color=["#C44E52" if d else "#4C72B0" for d in order["is_distractor"]])
    ax.axhline(baseline, color="k", ls=":", lw=1.0, label="random subspace r/d")
    ax.set_xticks(range(len(order)), order["category"], rotation=45, ha="right", fontsize=7)
    ax.set(title="Energy captured by the negation subspace (red = distractor)", ylabel="captured fraction")
    ax.legend(fontsize=7)
    fig.tight_layout()
    plt.show()
else:
    print("No distractor rows (`is_distractor` all False) — add `mis-` / `anti-` pairs to enable this check.")

## 10. Low-dimensional visualisations

In [ ]:
def pca_2d(X: np.ndarray, center=True):
    M = X - X.mean(axis=0, keepdims=True) if center else X
    U, s, Vt = np.linalg.svd(M, full_matrices=False)
    return M @ Vt[:2].T, (s[:2] ** 2) / (s ** 2).sum()


Z_delta, ev_delta = pca_2d(DELTA, center=False)
all_cats = sorted(set(CATS))
# Marker size and legend layout follow N and the number of categories: 10k points x 18 categories
# needs smaller, more transparent dots and a legend outside the axes.
dot_s, dot_a = (10, 0.45) if N > 2000 else (16, 0.75)
fig, ax = plt.subplots(1, 2, figsize=(11.5, 5.2))

for c in all_cats:
    m = CATS == c
    dist = bool(IS_DIST[m][0])
    ax[0].scatter(Z_delta[m, 0], Z_delta[m, 1], s=dot_s, alpha=dot_a, label=c + (" *" if dist else ""),
                  marker="x" if dist else "o")
ax[0].axhline(0, color="k", lw=0.5)
ax[0].axvline(0, color="k", lw=0.5)
ax[0].set(title=f"Δ vectors (PC1 {ev_delta[0]:.0%}, PC2 {ev_delta[1]:.0%}) — * = distractor",
          xlabel="PC1", ylabel="PC2")
ax[0].legend(fontsize=6, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.16),
             markerscale=1.6, handletextpad=0.3, columnspacing=0.9)

both = np.vstack([l2n(E_AFF), l2n(E_NEG)])
Z_both, ev_both = pca_2d(both)
half = len(E_AFF)
ax[1].scatter(Z_both[:half, 0], Z_both[:half, 1], s=dot_s, alpha=0.5, label="affirmative", color="#4C72B0")
ax[1].scatter(Z_both[half:, 0], Z_both[half:, 1], s=dot_s, alpha=0.5, label="negated", color="#C44E52")
for i in rng.choice(half, size=min(40, half), replace=False):
    ax[1].plot([Z_both[i, 0], Z_both[half + i, 0]], [Z_both[i, 1], Z_both[half + i, 1]],
               color="grey", lw=0.4, alpha=0.5)
ax[1].set(title=f"Sentence embeddings (PC1 {ev_both[0]:.0%}, PC2 {ev_both[1]:.0%})", xlabel="PC1", ylabel="PC2")
ax[1].legend(fontsize=7)
fig.tight_layout()
plt.show()

## 11. Layer sweep

Sammani et al. report that negation is most linearly separable in **intermediate** layers of the CLIP text
encoder. If the effective rank *also* reaches its minimum there, two independent methods agree that the
representation is cleanest mid-stack — which is where a transferable operator should be extracted.

The linear-probe accuracy replicates their diagnostic as a cross-check.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

LAYERS = list(range(0, N_TEXT_LAYERS + 1))
rows = []
for layer in LAYERS:
    a, b = get_embeddings(layer)
    dl = build_deltas(a, b)
    sp = spectrum_of(dl, center=False)
    mu = dl.mean(axis=0)
    share = float(mu @ mu / np.mean(np.sum(dl ** 2, axis=1)))

    X = np.vstack([l2n(a), l2n(b)])
    y = np.r_[np.zeros(len(a)), np.ones(len(b))]
    try:
        acc = float(cross_val_score(LogisticRegression(max_iter=2000, C=1.0), X, y,
                                    cv=min(5, int(min(np.bincount(y.astype(int))))), scoring="accuracy").mean())
    except Exception:
        acc = np.nan

    rows.append({"layer": layer, "translation_share": share,
                 "shannon_rank": shannon_effective_rank(sp.p),
                 "participation_ratio": participation_ratio(sp.eigenvalues),
                 "rank_tau90": rank_at_threshold(sp, 0.90), "probe_acc": acc})

LAYER_DF = pd.DataFrame(rows)
display(LAYER_DF.round(3))

fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.3))
ax[0].plot(LAYER_DF["layer"], LAYER_DF["translation_share"], "o-", ms=3.5, color="#4C72B0")
ax[0].set(title="Translation share ρ per layer", xlabel="layer", ylabel="ρ")
ax[1].plot(LAYER_DF["layer"], LAYER_DF["shannon_rank"], "o-", ms=3.5, label="Shannon", color="#DD8452")
ax[1].plot(LAYER_DF["layer"], LAYER_DF["participation_ratio"], "o--", ms=3.5, label="PR", color="#55A868")
ax[1].set(title="Effective rank per layer", xlabel="layer", ylabel="effective rank")
ax[1].legend(fontsize=7)
ax[2].plot(LAYER_DF["layer"], LAYER_DF["probe_acc"], "o-", ms=3.5, color="#C44E52")
ax[2].axhline(0.5, color="k", ls=":", lw=0.8)
ax[2].set(title="Linear probe accuracy (aff vs neg)", xlabel="layer", ylabel="CV accuracy", ylim=(0.4, 1.02))
fig.tight_layout()
plt.show()

best = LAYER_DF.loc[LAYER_DF["probe_acc"].idxmax()] if LAYER_DF["probe_acc"].notna().any() else None
if best is not None:
    print(f"Best probe layer = {int(best['layer'])} (acc {best['probe_acc']:.3f}, "
          f"Shannon rank {best['shannon_rank']:.2f}, ρ {best['translation_share']:.3f})")

## 12. Verdict

Maps the measured statistics onto H1–H4 and states what it implies for the correction operator and for the
cross-model transfer machinery. Thresholds live in `CONFIG` — they are deliberately explicit rather than
hidden inside the logic, because they encode a judgement call, not a fact.

In [ ]:
def verdict() -> dict:
    rho = translation_share
    r_sig = SIGNIFICANT_RANK
    intra, inter = MEAN_INTRA, MEAN_INTER
    has_types = (intra >= CONFIG["h2_min_intra_coherence"]) and (inter <= CONFIG["h2_max_inter_cosine"])

    # NOTE the ordering: H1 *is* a rank-1 outcome, so "nothing survives the null" must mean
    # strictly zero surviving components, otherwise a perfect translation gets misread as noise.
    if r_sig == 0:
        h, why = "H4?", "no component survives the null — check layers (§11) and the predictive test (§7) first"
    elif rho >= CONFIG["h1_translation_share"] and r_sig <= CONFIG["h1_max_rank"]:
        h, why = "H1", "the mean displacement dominates and almost no residual direction survives the null"
    elif r_sig <= CONFIG["h2_max_rank"] and has_types:
        h, why = "H2", "several significant components AND coherent, mutually distinct per-type directions"
    elif r_sig <= CONFIG["h2_max_rank"]:
        h, why = "H3", "several significant components but no per-type organisation — content-driven"
    else:
        h, why = "H4", "high significant rank: no low-dimensional linear structure"

    # A high translation share alongside a rank > 1 is a *mixed* regime: one shared displacement
    # plus genuine residual structure. That combination is both realistic and actionable.
    mixed = (h in {"H2", "H3"}) and (rho >= CONFIG["h1_translation_share"])

    plan = {
        "H1": ("single Householder reflection I − 2vvᵀ, no rule-based scope extraction",
               "classic orthogonal Procrustes on one vector"),
        "H2": ("soft mixture of per-type subspaces, weights = softmax of cosine to type centroids",
               "CCA + Procrustes on the subspace basis (reuse Agarwal's released pipeline)"),
        "H3": ("hybrid: global subspace projection + attention-weighted local content term",
               "CCA + Procrustes on the global part; local part recomputed natively per model"),
        "H4": ("try kernel PCA / a non-linear autoencoder bottleneck before abandoning the linear route",
               "autoencoder mapping (Oozeer et al.) rather than a linear map"),
    }[h.rstrip("?")]

    print("=" * 78)
    print(f"  VERDICT: {h}   —   {why}")
    print("=" * 78)
    print(f"  translation share rho          = {rho:.3f}")
    print(f"  significant rank (vs aff null) = {r_sig}")
    print(f"  significant rank (vs gaussian) = {sig_rank['gaussian']}")
    print(f"  Shannon effective rank         = {shannon_effective_rank(SPEC_UNCENTERED.p):.2f}")
    print(f"  participation ratio            = {participation_ratio(SPEC_UNCENTERED.eigenvalues):.2f}")
    print(f"  predictive rank (max excess R²)= {predictive_rank}")
    print(f"  intra-category coherence       = {intra:.3f}")
    print(f"  inter-category centroid cosine = {inter:.3f}")
    print("-" * 78)
    if mixed:
        print(f"  MIXED REGIME: rho = {rho:.2f} means a dominant shared translation coexists with")
        print(f"  {r_sig} significant residual directions. Model BOTH: subtract the mean displacement")
        print("  first, then apply the subspace operator to the residual.")
        print("-" * 78)
    print(f"  correction operator -> {plan[0]}")
    print(f"  transfer machinery  -> {plan[1]}")
    print("=" * 78)
    if DEMO:
        print('  (DEMO MODE — this verdict is meaningless; pick a real dataset via CONFIG["dataset"].)')

    return {"hypothesis": h, "translation_share": rho, "significant_rank": r_sig,
            "shannon_rank": shannon_effective_rank(SPEC_UNCENTERED.p),
            "participation_ratio": participation_ratio(SPEC_UNCENTERED.eigenvalues),
            "predictive_rank": predictive_rank, "intra_coherence": intra, "inter_cosine": inter}


RESULTS = verdict()

out_path = Path(CONFIG["cache_dir"]) / f"manip0_results_{DATASET}_{cache_key(CONFIG['layer'])}.json"
out_path.write_text(json.dumps(
    {"config": {k: str(v) for k, v in CONFIG.items()}, "dataset": DATASET,
     "n_pairs": int(N), "dim": int(D),
     "demo": DEMO, "results": {k: (float(v) if isinstance(v, (int, float, np.floating)) else v)
                               for k, v in RESULTS.items()}}, indent=2))
print(f"\nSaved -> {out_path}")

## Next steps

1. **Run both datasets and compare.** Flip `CONFIG["dataset"]` between `"taxonomy"` and `"nevir"` and put
   the two verdicts side by side. A rank that agrees across a synthetic, template-generated set and a
   real-world IR corpus is a property of negation; a rank that only appears on `taxonomy` is a property of
   the generator's templates.
2. **Close the two gaps in the taxonomy generator**: a content field (`object`) crossed with `category`,
   so §8b can separate genuine type structure (H2) from content leaking into Δ (H3), and `mis-`/`anti-`
   rows so §9 can check the subspace is not keying on a surface prefix cue. Both sections are already
   wired — they self-skip only because those columns are absent.
3. **Re-run per layer** and keep the layer where the probe peaks and the effective rank bottoms out.
4. **Involution test** (separate notebook): build the operator implied by the verdict
   (translation / projection / reflection) and check which one satisfies $f(f(x)) \approx x$ on real
   double negations — the theoretical question none of the nine reviewed papers tests.
5. **Replicate on a second, independently trained encoder.** If the effective rank matches before any
   alignment, the dimensionality of negation is a property of the task rather than of one architecture.
6. **Then, and only then**, build the cross-model transfer with the machinery the verdict selected.